In [1]:
import torch
import os
from dotenv import load_dotenv

%env LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
%env CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
%env LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
%env CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
%env PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"

!pip install -q pillow llama-index llama-cpp-python python-dotenv pandas jupyter pymupdf llama-index-embeddings-huggingface llama-index-llms-llama-cpp
!pip install -q -U google-genai


env: LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
env: CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
env: LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
env: CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
env: PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"


In [2]:
import os
from google import genai
from google.genai import types

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

if not GEMINI_API_KEY or GEMINI_API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("Please set GEMINI_API_KEY in your environment or assign it here.")

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized successfully!")


Gemini client initialized successfully!


In [3]:
import os
from llama_cpp import Llama

model_path = "content/mistral.gguf"
os.makedirs("content", exist_ok=True)
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

print(f"Model file ready: {model_path} ({os.path.getsize(model_path) / (1024 * 1024):.2f} MB)")


Model file ready: content/mistral.gguf (4166.07 MB)


In [4]:
import fitz  # PyMuPDF

pdf_path = "sample-sdf-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])
print(f"Extracted {len(text.split())} words across {len(doc)} page(s).")


Extracted 617 words across 3 page(s).


In [5]:
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

llm = LlamaCPP(
    model_path="content/mistral.gguf",
    temperature=0.1,
    max_new_tokens=512,
    context_window=2048,
    model_kwargs={"n_gpu_layers": 1}
)

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

documents = [Document(text=text)]
index = VectorStoreIndex.from_documents(documents)
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(response_mode="tree_summarize")
mistral_engine = RetrieverQueryEngine(retriever=retriever, response_synthesizer=response_synthesizer)


ggml_metal_device_init: testing tensor API for f16 support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x101149710 | th_max = 1024 | th_width =   32
ggml_metal_device_init: testing tensor API for bfloat support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x101149710 | th_max = 1024 | th_width =   32
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa_aux' library in 0.196 sec
ggml_metal_library_compile_all: compiled 'fa_f16' library in 1.375 sec
ggml_metal_library_compile_all: compiled 'fa_f32' library in 2.960 sec
ggml_metal_library_compile_all: compiled 'fa_q4_0' library in 0.852 sec
ggml_metal_library_compile_all: compiled 'fa_q4_1' library in 0.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
import time
import pandas as pd

queries = [
    "What sterilization method was used for this product?",
    "Summarize the key specifications in this document.",
    "What are the storage conditions for this product?"
]

def query_gemini(q: str, context: str):
    prompt = f"Context from SDF Document:\n{context}\n\nQuestion: {q}\n\nAnswer:"
    t0 = time.perf_counter()
    res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    elapsed = time.perf_counter() - t0
    return res.text.strip(), elapsed

benchmark_rows = []

for q in queries:
    print(f"\nEvaluating: {q}")
    
    # Test Mistral
    t0_m = time.perf_counter()
    m_resp = mistral_engine.query(q)
    lat_m = time.perf_counter() - t0_m
    m_text = str(m_resp).strip()
    
    # Test Gemini
    g_text, lat_g = query_gemini(q, text)
    
    benchmark_rows.append({
        "Query": q,
        "Mistral Latency (s)": round(lat_m, 2),
        "Gemini Latency (s)": round(lat_g, 2),
        "Mistral Length (words)": len(m_text.split()),
        "Gemini Length (words)": len(g_text.split()),
        "Mistral Output": m_text,
        "Gemini Output": g_text
    })

df_results = pd.DataFrame(benchmark_rows)
display(df_results[["Query", "Mistral Latency (s)", "Gemini Latency (s)", "Mistral Length (words)", "Gemini Length (words)"]])



Evaluating: What sterilization method was used for this product?


ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_bin_fuse_f32_f32_f32_4', name = 'kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0'
ggml_metal_library_compile_pipeline: loaded kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0     0x772e29dc00 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_rms_norm_mul_f32_4', name = 'kernel_rms_norm_mul_f32_4'
ggml_metal_library_compile_pipeline: loaded kernel_rms_norm_mul_f32_4                    0x772e29df80 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_mul_mv_q6_K_f32', name = 'kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0'
ggml_metal_library_compile_pipeline: loaded kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0     0x772e29e300 | th_max = 1024 | th_width =   32
llama_perf_context_print:        load time =   12558.68 ms
llama_perf_context_print: prompt eval time =   12557.79 ms /  1286 tokens 


Evaluating: Summarize the key specifications in this document.


Llama.generate: 1270 prefix-match hit, remaining 16 prompt tokens to eval
llama_perf_context_print:        load time =   12558.68 ms
llama_perf_context_print: prompt eval time =     220.57 ms /    16 tokens (   13.79 ms per token,    72.54 tokens per second)
llama_perf_context_print:        eval time =   18900.09 ms /   458 runs   (   41.27 ms per token,    24.23 tokens per second)
llama_perf_context_print:       total time =   19225.23 ms /   474 tokens
llama_perf_context_print:    graphs reused =        456



Evaluating: What are the storage conditions for this product?


Llama.generate: 1270 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =   12558.68 ms
llama_perf_context_print: prompt eval time =     424.07 ms /    14 tokens (   30.29 ms per token,    33.01 tokens per second)
llama_perf_context_print:        eval time =    8019.96 ms /   193 runs   (   41.55 ms per token,    24.06 tokens per second)
llama_perf_context_print:       total time =    8471.69 ms /   207 tokens
llama_perf_context_print:    graphs reused =        192


,Query,Mistral Latency (s),Gemini Latency (s),Mistral Length (words),Gemini Length (words)
0,What sterilization method was used for this pr...,14.53,4.99,25,35
1,Summarize the key specifications in this docum...,19.30,10.28,244,261
2,What are the storage conditions for this product?,8.73,4.15,118,36


In [7]:
# Inspect full text responses side-by-side
for idx, row in df_results.iterrows():
    print(f"=== Question: {row['Query']} ===")
    print(f"\n--- Mistral (Time: {row['Mistral Latency (s)']}s) ---\n{row['Mistral Output']}")
    print(f"\n--- Gemini (Time: {row['Gemini Latency (s)']}s) ---\n{row['Gemini Output']}\n\n")


=== Question: What sterilization method was used for this product? ===

--- Mistral (Time: 14.53s) ---
The product was sterilized using autoclave and gamma irradiation methods. The autoclave sterilization was for the pump tubing, while gamma irradiation was for the inlets.

--- Gemini (Time: 4.99s) ---
Based on the provided Certificates of Quality:

The sterilization methods used for the products are:
*   **Autoclave** for the pump tubing (121°C > 15 min).
*   **Gamma Irradiation** for the inlets (25.0 – 40.0 kGy).


=== Question: Summarize the key specifications in this document. ===

--- Mistral (Time: 19.3s) ---
1. The recommended storage temperature for ÄKTA ready flow kits is > +5 C, but they can be allowed to acclimate to a warmer temperature before use to reduce the risk of damage.
2. The operating temperature for ÄKTA ready flow kits is +2 C to +40 C.
3. The two ÄKTA ready flow kits mentioned in the document are the High Flow Kit F, Modified (Part Number: 29477427) and the High